# Entity-level recall (A14)

Address-level recall can hide operators who keep most of their payout. This notebook evaluates the saved test
predictions of `03` (XGBoost), `04` (LightGBM) and `06` (cross-ensemble) at the entity level. No model is
trained.

| Entity | Group key | Note |
|---|---|---|
| E1 gas provision tree | `provision_tree` | Under the group split a tree's Sybils are all in one partition. Most IxL Sybils are single-wallet trees. |
| E2 bounty report | `commonwealth report link` | A report's Sybils can span partitions; metrics use its test Sybils. Also restricted to reports with every Sybil in test. |
| E3 reporter | `reward address1` | Coarse; descriptive only. |

Metrics per entity type, model and threshold (the model's validation threshold, and 0.95, 0.90, 0.80, 0.70),
overall and by entity size (test Sybils per entity: 1, 2–5, 6–20, 21–100, >100): any-hit, majority-hit and
full-hit recall; within-entity recall (unweighted and Sybil-weighted); the share of the test Sybils'
`allocation` (ZRO) left on unflagged addresses; and the entities with at least half their allocation unflagged
("partly detected but still profitable").

**Inputs.** The predictions are read from `output/`, which is not committed. The notebook recomputes each model's
validation threshold and test metrics from them and records whether they equal the committed `results/` of
`03`, `04` and `06`. If they do not (for example, predictions regenerated on a machine whose XGBoost and LightGBM
builds give slightly different trees), the run is flagged in the results file, and the acceptance check is made
against the metrics of the predictions actually used.

**Caveat.** A tree or a report is a proxy for an operator, not the operator: one operator can fund many wallets
directly from an exchange, giving many single-wallet trees. `allocation` is the intended payout, not profit (it
excludes the operator's costs).

**Results**: `results/14_entity_level_recall.json`.

## Setup and inputs

In [1]:
import platform, warnings
import numpy as np
import pandas as pd
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
DATA_DIR = './data'

NBS = ['03_xgboost_sybil', '04_lightgbm_sybil', '06_cross_ensemble_sybil']
prov = sp.provenance(NBS)
print(prov.to_string(index=False))
assert prov['dependencies_unchanged'].all()

# model -> (notebook, probability column, name of its entry in the notebook's results)
MODELS = {'XGBoost': ('03_xgboost_sybil', 'prob_xgb', 'XGBoost (tuned, 3-seed)'),
          'LightGBM': ('04_lightgbm_sybil', 'prob_lgbm', 'LightGBM (tuned, 3-seed)'),
          'Ensemble': ('06_cross_ensemble_sybil', 'prob_ens', 'Cross-Ensemble')}
P = {}
for part in ['val', 'test']:
    frames = {m: sp.load_predictions(nb, part) for m, (nb, _, _) in MODELS.items()}
    base = frames['XGBoost'][['addr', 'category', 'y']]
    for m, (_, col, _) in MODELS.items():
        assert (frames[m]['addr'].to_numpy() == base['addr'].to_numpy()).all()   # same rows, same order
        assert (frames[m]['y'].to_numpy() == base['y'].to_numpy()).all()
    P[part] = base.assign(**{col: frames[m][col].to_numpy() for m, (_, col, _) in MODELS.items()})
print(f"Validation {len(P['val']):,} rows, test {len(P['test']):,} rows")

Code commit: 8188899


               notebook  commit  dependencies_unchanged
       03_xgboost_sybil 622ca58                    True
      04_lightgbm_sybil 71a009e                    True
06_cross_ensemble_sybil 93a79f5                    True


Validation 91,305 rows, test 130,435 rows


### Thresholds, and whether the predictions match the committed results

Each model's threshold is the F1-maximizing threshold on validation (`sp.best_f1_threshold`), the rule `03`,
`04` and `06` use; it is compared with the threshold and test metrics in their committed `results/`.

In [2]:
THR, inputs = {}, {}
for m, (nb, col, name) in MODELS.items():
    r = sp.evaluate(name, P['val']['y'], P['val'][col].to_numpy(), P['test']['y'], P['test'][col].to_numpy())
    committed = next(x for x in sp.load_results(nb)['models'] if x['name'] == name)
    keys = ['threshold', 'precision', 'recall', 'f1', 'auroc', 'ap']
    match = all(abs(r[k] - committed[k]) < 1e-9 for k in keys)
    THR[m] = r['threshold']
    inputs[m] = dict(notebook=nb, matches_committed=match,
                     used={k: float(r[k]) for k in keys}, committed={k: committed[k] for k in keys},
                     committed_code_commit=sp.load_results(nb)['code_commit'])
INPUTS_MATCH_COMMITTED = all(v['matches_committed'] for v in inputs.values())
platform_info = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                     python=platform.python_version())
print(pd.DataFrame([dict(model=m, matches_committed=v['matches_committed'],
                         **{f'{k}_used': v['used'][k] for k in ['threshold', 'recall', 'f1']},
                         **{f'{k}_committed': v['committed'][k] for k in ['threshold', 'recall', 'f1']})
                    for m, v in inputs.items()]).round(4).to_string(index=False))
print('Platform:', platform_info)
if not INPUTS_MATCH_COMMITTED:
    print('\nFLAG: the predictions in output/ do not reproduce the committed results of 03, 04 and 06. '
          'Entity metrics below describe the predictions actually used; the acceptance check is made against them.')

   model  matches_committed  threshold_used  recall_used  f1_used  threshold_committed  recall_committed  f1_committed
 XGBoost               True          0.6420       0.7216   0.7159               0.6420            0.7216        0.7159
LightGBM               True          0.5444       0.7276   0.7149               0.5444            0.7276        0.7149
Ensemble               True          0.6055       0.7229   0.7165               0.6055            0.7229        0.7165
Platform: {'machine': 'x86_64', 'processor': 'x86_64', 'system': 'Linux', 'python': '3.11.15'}


### Gas provision trees and bounty-report fields

In [3]:
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
fcfs = pd.read_csv(sp.data_paths(DATA_DIR)['sybil'])
fcfs.columns = fcfs.columns.str.lower().str.strip()
fcfs['address'] = fcfs['address'].str.lower()
assert fcfs['address'].is_unique
FIELDS = {'commonwealth report link': 'report', 'github issue': 'github_issue',
          'reward address1': 'reporter', 'allocation': 'allocation'}

T = P['test'].merge(df[['addr', 'provision_tree', 'sybil']], on='addr', how='left', validate='one_to_one')
assert (T['y'] == T['sybil']).all()
T = T.merge(fcfs[['address', *FIELDS]].rename(columns={'address': 'addr', **FIELDS}), on='addr', how='left',
            validate='one_to_one')
syb = T[T['y'] == 1].copy()
assert syb['report'].notna().all() and syb['allocation'].notna().all()   # every test Sybil matched
assert T.loc[T['y'] == 0, 'report'].isna().all()
N_TEST_SYB = len(syb)
assert N_TEST_SYB == 5_463
ALLOC_TOTAL = float(syb['allocation'].sum())
print(f"{N_TEST_SYB:,} test Sybils; allocation {ALLOC_TOTAL:,.1f} ZRO; "
      f"{syb['provision_tree'].nunique():,} trees, {syb['report'].nunique()} reports, {syb['reporter'].nunique()} reporters")

5,463 test Sybils; allocation 32,696.7 ZRO; 4,907 trees, 239 reports, 132 reporters


### Coverage of each entity type

In [4]:
all_syb = df[df['sybil'] == 1].assign(report=lambda d: d['addr'].map(fcfs.set_index('address')['commonwealth report link']))
tree_rows = df.groupby('provision_tree').size()

# E1: a test tree's Sybils are all in test (group split)
tree_syb_all = all_syb.groupby('provision_tree').size()
tree_syb_test = syb.groupby('provision_tree').size()
assert (tree_syb_test == tree_syb_all.loc[tree_syb_test.index]).all()
single_tree = tree_rows.loc[tree_syb_test.index] == 1
ixl = syb['category'] == 'IxL'
coverage = dict(
    E1=dict(entities=int(len(tree_syb_test)), all_sybils_in_test=True,
            single_wallet_tree_share=float(single_tree.mean()),
            ixl_test_sybils_in_single_wallet_trees_share=float((tree_rows.loc[syb.loc[ixl, 'provision_tree']] == 1).mean())),
)
# E2: reports can span partitions
rep_all = all_syb.groupby('report').size()
rep_test = syb.groupby('report').size()
full_reports = rep_test.index[rep_test.to_numpy() == rep_all.loc[rep_test.index].to_numpy()]
coverage['E2'] = dict(entities=int(len(rep_test)), reports_total=int(len(rep_all)),
                      reports_with_all_sybils_in_test=int(len(full_reports)),
                      test_sybils_in_those_reports=int(rep_test.loc[full_reports].sum()),
                      test_share_of_report_sybils_median=float((rep_test / rep_all.loc[rep_test.index]).median()))
coverage['E3'] = dict(entities=int(syb['reporter'].nunique()))
for k, v in coverage.items():
    print(k, {a: (round(b, 4) if isinstance(b, float) else b) for a, b in v.items()})

E1 {'entities': 4907, 'all_sybils_in_test': True, 'single_wallet_tree_share': 0.9715, 'ixl_test_sybils_in_single_wallet_trees_share': 0.9853}
E2 {'entities': 239, 'reports_total': 311, 'reports_with_all_sybils_in_test': 18, 'test_sybils_in_those_reports': 20, 'test_share_of_report_sybils_median': 0.3112}
E3 {'entities': 132}


## Entity metrics

In [5]:
BUCKETS = ['1', '2–5', '6–20', '21–100', '>100']
ENTITIES = {'E1 gas provision tree': ('provision_tree', None),
            'E2 bounty report': ('report', None),
            'E2 bounty report (all Sybils in test)': ('report', set(full_reports)),
            'E3 reporter': ('reporter', None)}
LEVELS = [0.95, 0.90, 0.80, 0.70]

def entity_table(sub, key, flagged):
    """One row per entity: test Sybils, flagged Sybils, allocation and unflagged allocation."""
    s = sub.assign(flag=flagged, alloc_unflagged=np.where(flagged, 0.0, sub['allocation']))
    g = s.groupby(key).agg(n=('flag', 'size'), hit=('flag', 'sum'), alloc=('allocation', 'sum'),
                           alloc_unflagged=('alloc_unflagged', 'sum'))
    g['bucket'] = pd.cut(g['n'], [0, 1, 5, 20, 100, np.inf], labels=BUCKETS).astype(str)
    return g

def metrics(g):
    frac = g['hit'] / g['n']
    unflag = g['alloc_unflagged'] / g['alloc']
    return dict(entities=len(g), sybils=int(g['n'].sum()),
                any_hit=float((g['hit'] >= 1).mean()), majority_hit=float((frac >= 0.5).mean()),
                full_hit=float((frac == 1).mean()),
                within_recall_unweighted=float(frac.mean()), within_recall_weighted=float(g['hit'].sum() / g['n'].sum()),
                allocation=float(g['alloc'].sum()),
                residual_allocation_share=float(g['alloc_unflagged'].sum() / g['alloc'].sum()),
                entities_majority_allocation_unflagged=int((unflag >= 0.5).sum()),
                entities_majority_allocation_unflagged_share=float((unflag >= 0.5).mean()))

rows, dist = [], {}
for ent, (key, keep) in ENTITIES.items():
    sub = syb if keep is None else syb[syb[key].isin(keep)]
    for m, (_, col, _) in MODELS.items():
        for label, thr in [('validation', THR[m])] + [(f'{t:.2f}', t) for t in LEVELS]:
            g = entity_table(sub, key, (sub[col] >= thr).to_numpy())
            if keep is None:
                assert g['n'].sum() == N_TEST_SYB                         # every test Sybil in one entity
                assert abs(g['alloc'].sum() - ALLOC_TOTAL) < 1e-6 * ALLOC_TOTAL
            rows.append(dict(entity=ent, model=m, threshold_label=label, threshold=thr, bucket='all', **metrics(g)))
            for b in BUCKETS:
                gb = g[g['bucket'] == b]
                if len(gb):
                    rows.append(dict(entity=ent, model=m, threshold_label=label, threshold=thr, bucket=b, **metrics(gb)))
    n = entity_table(sub, key, np.zeros(len(sub), bool))['n']
    dist[ent] = dict(entities=int(len(n)), mean=float(n.mean()), median=float(n.median()), max=int(n.max()),
                     by_bucket={b: int(c) for b, c in pd.cut(n, [0, 1, 5, 20, 100, np.inf], labels=BUCKETS)
                                .value_counts().reindex(BUCKETS).items()})
flat = pd.DataFrame(rows)
print(pd.DataFrame(dist).T[['entities', 'mean', 'median', 'max']].to_string())
print(pd.DataFrame({e: d['by_bucket'] for e, d in dist.items()}).T.to_string())

                                      entities       mean median  max
E1 gas provision tree                     4907   1.113308    1.0  100
E2 bounty report                           239  22.857741    7.0  294
E2 bounty report (all Sybils in test)       18   1.111111    1.0    2
E3 reporter                                132  41.386364   15.0  453
                                          1  2–5  6–20  21–100  >100
E1 gas provision tree                  4811   69    22       5     0
E2 bounty report                         56   51    71      49    12
E2 bounty report (all Sybils in test)    16    2     0       0     0
E3 reporter                               9   18    49      43    13


### Acceptance checks

In [6]:
e1 = flat[(flat['entity'] == 'E1 gas provision tree') & (flat['threshold_label'] == 'validation') & (flat['bucket'] == 'all')]
for m in MODELS:
    w = float(e1.loc[e1['model'] == m, 'within_recall_weighted'].iloc[0])
    assert abs(w - inputs[m]['used']['recall']) < 1e-9          # address-level recall of the predictions used
    if INPUTS_MATCH_COMMITTED:
        assert abs(w - inputs[m]['committed']['recall']) < 1e-9  # ... which are the committed results
    print(f"{m:<9} E1 Sybil-weighted recall {w:.6f} = address-level recall {inputs[m]['used']['recall']:.6f}"
          f" (committed {inputs[m]['committed']['recall']:.6f})")
for ent in ['E1 gas provision tree', 'E2 bounty report', 'E3 reporter']:
    assert (flat.loc[(flat['entity'] == ent) & (flat['bucket'] == 'all'), 'sybils'] == N_TEST_SYB).all()
print('Entity Sybil counts sum to', N_TEST_SYB, 'and allocation totals to', round(ALLOC_TOTAL, 3), 'for every entity type')

XGBoost   E1 Sybil-weighted recall 0.721582 = address-level recall 0.721582 (committed 0.721582)
LightGBM  E1 Sybil-weighted recall 0.727622 = address-level recall 0.727622 (committed 0.727622)
Ensemble  E1 Sybil-weighted recall 0.722863 = address-level recall 0.722863 (committed 0.722863)
Entity Sybil counts sum to 5463 and allocation totals to 32696.72 for every entity type


## Summary at the validation threshold (LightGBM and ensemble)

In [7]:
show = ['entities', 'sybils', 'any_hit', 'majority_hit', 'full_hit', 'within_recall_unweighted',
        'within_recall_weighted', 'residual_allocation_share', 'entities_majority_allocation_unflagged',
        'entities_majority_allocation_unflagged_share']
for ent in ENTITIES:
    t = flat[(flat['entity'] == ent) & (flat['threshold_label'] == 'validation') & flat['model'].isin(['LightGBM', 'Ensemble'])]
    print(f'=== {ent}')
    print(t.set_index(['model', 'bucket'])[show].round(4).to_string(), '\n')

=== E1 gas provision tree
                 entities  sybils  any_hit  majority_hit  full_hit  within_recall_unweighted  within_recall_weighted  residual_allocation_share  entities_majority_allocation_unflagged  entities_majority_allocation_unflagged_share
model    bucket                                                                                                                                                                                                                      
LightGBM all         4907    5463   0.7756        0.7754    0.7742                    0.7750                  0.7276                     0.3677                                    1103                                        0.2248
         1           4811    4811   0.7847        0.7847    0.7847                    0.7847                  0.7847                     0.2571                                    1036                                        0.2153
         2–5           69     193   0.4058        0.39

### All thresholds, all entities (bucket = all)

In [8]:
print(flat[flat['bucket'] == 'all'].set_index(['entity', 'model', 'threshold_label'])[
    ['threshold', 'any_hit', 'majority_hit', 'full_hit', 'within_recall_weighted', 'residual_allocation_share',
     'entities_majority_allocation_unflagged']].round(4).to_string())

                                                                threshold  any_hit  majority_hit  full_hit  within_recall_weighted  residual_allocation_share  entities_majority_allocation_unflagged
entity                                model    threshold_label                                                                                                                                       
E1 gas provision tree                 XGBoost  validation          0.6420   0.7673        0.7671    0.7652                  0.7216                     0.3704                                    1146
                                               0.95                0.9500   0.5482        0.5476    0.5466                  0.5006                     0.5715                                    2222
                                               0.90                0.9000   0.6313        0.6307    0.6297                  0.5826                     0.4973                                    1815
          

## Save results

In [9]:
sp.save_results([], '14_entity_level_recall', extra=dict(
    inputs=dict(predictions_match_committed_results=INPUTS_MATCH_COMMITTED, platform=platform_info, models=inputs,
                note='Predictions read from output/ (not committed). If predictions_match_committed_results is false, '
                     'they were regenerated by 03, 04 and 06 on this platform and differ from the committed results; '
                     'every metric here describes the predictions actually used.'),
    thresholds=dict(validation=THR, fixed=LEVELS), buckets=BUCKETS, coverage=coverage, distribution=dist,
    table=flat.to_dict('records'),
    caveat='A gas provision tree or a bounty report is a proxy for an operator, not the operator: one operator can '
           'fund many wallets directly from an exchange, giving many single-wallet trees. allocation is the intended '
           'payout (ZRO), not profit; it excludes the operator\'s costs.'))

Saved results/14_entity_level_recall.json
